# Docker Compose: Multi-Container Projects


## Learning Goals

By the end of this notebook, you should be able to:

- explain why Docker Compose exists
- define multiple services in one `compose.yaml` file
- run multiple containers as one project
- understand service names like `redis` and `db`
- run commands inside a service with `docker compose exec`
- understand basic volumes for persistent data

In this session we stay beginner-friendly. We use simple Python scripts, Redis, and PostgreSQL. Django deployment with Docker is a later topic.


## Why Docker Compose?

Real backend projects usually need more than one container.

Example project:

```text
app container       → Python/Django/DRF app
redis container     → Redis cache or queue broker
db container        → PostgreSQL database
```

So Dockerizing a project does not always mean:

```text
one project = one container
```

More realistic:

```text
one project = multiple containers working together
```

Running each container manually with `docker run` becomes hard.

Docker Compose lets us describe all services in one YAML file and run them together.


## Compose File Structure

Recommended modern file name:

```text
compose.yaml
```

Older projects often use:

```text
docker-compose.yml
```

Basic structure:

```yaml
services:
  app:
    image: my_python_app
    ports:
      - "5000:5000"
```

Run:

```bash
docker compose up -d
```

Older command style:

```bash
docker-compose up -d
```

In this course, we use the modern `docker compose` command.


## Common Compose Concepts

| Concept | Meaning |
|---------|---------|
| `services` | Containers that make up the project. |
| `image` | Use an existing image. |
| `build` | Build an image from a Dockerfile. |
| `ports` | Publish container ports to the host. |
| `environment` | Set environment variables. |
| `env_file` | Load variables from a file. |
| `volumes` | Persist or share data. |
| `depends_on` | Start one service before another. |

Compose creates a default network. Services can reach each other by service name.


## Basic Commands

Start services:

```bash
docker compose up -d
```

Build and start:

```bash
docker compose up --build
```

View logs:

```bash
docker compose logs -f
```

View service status:

```bash
docker compose ps
```

Run a command inside a service:

```bash
docker compose exec SERVICE_NAME COMMAND
```

Stop and remove containers/network:

```bash
docker compose down
```

Stop and remove containers/network/volumes:

```bash
docker compose down -v
```

Be careful: `down -v` removes volumes and can delete database data.


## Main Sample: Python Logger + Redis

Files:

```text
session45/samples/compose_redis_logger/
    compose.yaml
    Dockerfile
    logger.py
    requirements.txt
```

Goal:

```text
logger container → writes a log message → redis container
```

This teaches the most important Compose idea:

> Different containers can talk to each other by service name.


## Redis Logger Compose File

`compose.yaml`:

```yaml
services:
  logger:
    build: .
    environment:
      REDIS_HOST: redis
      REDIS_PORT: 6379
    depends_on:
      - redis

  redis:
    image: redis:7-alpine
```

Important points:

| Part | Meaning |
|------|---------|
| `logger` | Our Python app service. |
| `build: .` | Build the logger image from the local Dockerfile. |
| `redis` | Redis service. |
| `image: redis:7-alpine` | Use the official Redis image. |
| `REDIS_HOST: redis` | The Python app connects to hostname `redis`. |
| `depends_on` | Start Redis before logger. |

`redis` is not a random hostname. It is the Compose service name.


## Redis Logger Python Code

`logger.py` connects to Redis and stores one log message:

```python
import os
from datetime import datetime

import redis

client = redis.Redis(
    host=os.getenv("REDIS_HOST", "redis"),
    port=int(os.getenv("REDIS_PORT", "6379")),
    decode_responses=True,
)

message = f"log created at {datetime.now().isoformat(timespec='seconds')}"
client.rpush("logs", message)
print(message)
```

The actual sample file also retries a few times because Redis may need a moment to become ready.


## Run the Redis Logger

Go to the sample directory:

```bash
cd session45/samples/compose_redis_logger
```

Build and run:

```bash
docker compose up --build
```

You should see the logger print something like:

```text
log created at 2026-08-14T12:30:10
```

The logger container writes the data into the Redis container.


## Inspect Redis Data

Open another terminal in the same directory and run:

```bash
docker compose exec redis redis-cli LRANGE logs 0 -1
```

This means:

| Part | Meaning |
|------|---------|
| `docker compose exec` | Run a command inside a running Compose service. |
| `redis` | The service name. |
| `redis-cli` | Redis command-line tool inside the Redis container. |
| `LRANGE logs 0 -1` | Show all values in the `logs` list. |

Run the logger again:

```bash
docker compose run --rm logger
```

Check Redis again:

```bash
docker compose exec redis redis-cli LRANGE logs 0 -1
```


## Stop the Redis Logger Project

Stop and remove the containers/network:

```bash
docker compose down
```

Because this Redis example does not define a volume, its data is temporary.

That is fine for the first Compose example. It keeps the lesson simple.


## Why `redis`, not `localhost`?

Inside the logger container:

```text
localhost = the logger container itself
```

Redis is running in another container, so the logger should connect to:

```text
redis:6379
```

because the Redis service is named `redis`.

Mental model:

```text
logger container ── connects to hostname redis ──► redis container
```

This same idea later applies to PostgreSQL:

```text
app container ── connects to hostname db ──► db container
```


## Environment Variables in Compose

In the Redis logger example:

```yaml
environment:
  REDIS_HOST: redis
  REDIS_PORT: 6379
```

These values become environment variables inside the logger container.

Python reads them with:

```python
os.getenv("REDIS_HOST")
os.getenv("REDIS_PORT")
```

This is useful because we can change configuration without changing Python code.


## Optional Exercise: Python Logger + PostgreSQL

PostgreSQL is more complex than Redis, so treat this as an exercise or curious extra.

Files:

```text
session45/samples/compose_postgres_logger/
    compose.yaml
    Dockerfile
    logger.py
    requirements.txt
```

Goal:

```text
logger container → inserts a row → PostgreSQL container
```

This teaches:

- PostgreSQL service container
- database environment variables
- named volume for persistent database data
- inspecting data with `psql`


## PostgreSQL Logger Compose File

`compose.yaml`:

```yaml
services:
  logger:
    build: .
    environment:
      POSTGRES_HOST: db
      POSTGRES_PORT: 5432
      POSTGRES_DB: logsdb
      POSTGRES_USER: user
      POSTGRES_PASSWORD: password
    depends_on:
      - db

  db:
    image: postgres:15-alpine
    environment:
      POSTGRES_DB: logsdb
      POSTGRES_USER: user
      POSTGRES_PASSWORD: password
    volumes:
      - pgdata:/var/lib/postgresql/data

volumes:
  pgdata:
```

Important part:

```text
POSTGRES_HOST=db
```

The logger connects to the PostgreSQL service by its Compose service name: `db`.


## Run the PostgreSQL Exercise

Go to the sample directory:

```bash
cd session45/samples/compose_postgres_logger
```

Build and run:

```bash
docker compose up --build
```

Inspect database rows from another terminal:

```bash
docker compose exec db psql -U user -d logsdb -c "SELECT id, message, created_at FROM logs;"
```

Run the logger again:

```bash
docker compose run --rm logger
```

Check rows again:

```bash
docker compose exec db psql -U user -d logsdb -c "SELECT id, message, created_at FROM logs;"
```

Stop containers but keep database volume:

```bash
docker compose down
```

Stop containers and delete database volume:

```bash
docker compose down -v
```

Warning: `down -v` deletes the PostgreSQL data volume.


## Volumes and Data Persistence

Redis example:

```text
no volume → data is temporary
```

PostgreSQL example:

```yaml
volumes:
  - pgdata:/var/lib/postgresql/data
```

This means PostgreSQL stores data in a named Docker volume.

View volumes:

```bash
docker volume ls
```

Remove a volume:

```bash
docker volume rm VOLUME_NAME
```

Warning:

```bash
docker compose down -v
```

removes Compose volumes too. This can delete your database data.


## Scaling Services

Compose can start multiple containers for one service:

```bash
docker compose up --scale logger=3
```

Beginner note:

Scaling is not just “run more containers.” You also need to think about duplicate work, load balancing, and shared data.

For now, just know that Compose can scale services, but we will not go deep.


## Curious Note: Beyond Compose

Docker Compose is good for learning, local development, and simple single-server deployments.

Big companies may use orchestration tools when they have many containers across many servers.

Examples:

- Kubernetes
- Docker Swarm
- cloud container platforms

These tools help with scaling, rolling updates, self-healing, and load balancing.

For this course, just remember the name **Kubernetes** as the advanced next step. Docker Compose is enough for now.


## Summary

- Docker Compose manages multi-container projects.
- A Compose project has services such as `logger`, `redis`, and `db`.
- Containers can talk to each other by service name.
- Use environment variables to configure service addresses.
- Use `docker compose exec` to inspect running services.
- Redis is a good first example for container-to-container communication.
- PostgreSQL adds persistent data and named volumes.
- `docker compose down -v` can delete data volumes.
- Compose is a step before advanced orchestration tools such as Kubernetes.
